# 04-1. 출시 후 LLM 분류 리뷰 전처리

이 노트북은 `04_run_llm_postlaunch_tainted-grail_analysis.ipynb`에서 생성한 LLM 분류 결과를 읽어, 후속 단계인 **LLM 기반 패치·운영 전략 제안**에 사용할 근거 데이터를 만든다.

흐름은 출시 전 분석의 `03-1`과 맞춘다.

```text
04. 리뷰 LLM 분류
→ 04-1. 분류된 리뷰 전처리
→ 04-2. LLM 기반 패치·운영 전략 제안
```

주의할 점은 다음과 같다.

- 이 노트북에서는 LLM을 호출하지 않는다.
- LLM이 분류한 감정, 이슈 태그, urgency 후보를 그대로 최종 우선순위로 사용하지 않는다.
- 패치·운영 우선 검토 수준은 **이슈 반복 수, 부정·혼합 리뷰 수, Steam 비추천 리뷰 수, 최근 리뷰 반복 여부, 짧은 플레이타임 부정 반응**을 기준으로 규칙 기반 계산한다.
- `High urgency`는 04번 리뷰 분류 단계에서 LLM이 판단한 시급도 후보이므로, **우선순위 계산에는 직접 사용하지 않고 보조 설명 지표로만 유지한다.**
- 04-2에서는 이 노트북에서 계산한 `action_group_hint`, `rule_priority_hint`, `priority_rule_detail`, `priority_reason`을 고정 근거로 사용한다.


# 1. 기본 설정

In [1]:
# ============================================================
# 기본 라이브러리
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)


In [2]:
# ============================================================
# 경로 설정
# ============================================================
# 다른 환경에서 실행할 경우 ROOT만 본인 프로젝트 경로에 맞게 수정한다.
ROOT = Path.cwd()

# Jupyter 실행 위치가 하위 폴더일 수 있으므로,
# data/preprocessed 폴더가 보일 때까지 상위 폴더를 탐색한다.
if not (ROOT / "data" / "preprocessed").exists():
    for parent in ROOT.parents:
        if (parent / "data" / "preprocessed").exists():
            ROOT = parent
            break

# 04_run_llm_postlaunch_tainted-grail_analysis.ipynb에서 사용한 실행 이름과 맞춘다.
RUN_NAME = "postlaunch_tainted-grail"
OUTPUT_DIR = ROOT / "data" / "outputs" / RUN_NAME

# 04번 LLM 실행 산출물
RESULT_CSV_PATH = OUTPUT_DIR / "llm_review_analysis_result.csv"
ISSUE_TAG_FLAT_PATH = OUTPUT_DIR / "llm_issue_tags_flat.csv"
LLM_INPUT_PATH = OUTPUT_DIR / "llm_input_reviews.csv"

# 이번 04-1 노트북에서 생성할 CSV 저장 폴더
POSTLAUNCH_PREPROCESS_DIR = OUTPUT_DIR / "postlaunch_preprocess_data"
POSTLAUNCH_PREPROCESS_DIR.mkdir(parents=True, exist_ok=True)

# 04-1 전처리 산출 파일
# 산출물은 04-2에서 실제로 사용할 최소 파일만 저장한다.
POSTLAUNCH_REVIEW_BASE_PATH = POSTLAUNCH_PREPROCESS_DIR / "postlaunch_review_base.csv"
POSTLAUNCH_ISSUE_SUMMARY_PATH = POSTLAUNCH_PREPROCESS_DIR / "postlaunch_issue_summary.csv"
POSTLAUNCH_PATCH_OPS_EVIDENCE_BASE_PATH = POSTLAUNCH_PREPROCESS_DIR / "postlaunch_patch_ops_evidence_base.csv"
TABLEAU_POSTLAUNCH_SOURCE_PATH = POSTLAUNCH_PREPROCESS_DIR / "tableau_postlaunch_patch_ops_source.csv"

print("ROOT:", ROOT)
print("RUN_NAME:", RUN_NAME)
print("LLM 결과 폴더:", OUTPUT_DIR)
print("전처리 저장 폴더:", POSTLAUNCH_PREPROCESS_DIR)
print("LLM 리뷰 결과:", RESULT_CSV_PATH)
print("LLM 이슈 태그 결과:", ISSUE_TAG_FLAT_PATH)

ROOT: c:\Users\joon5\Documents\github\steam-indie-game-analysis
RUN_NAME: postlaunch_tainted-grail
LLM 결과 폴더: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_tainted-grail
전처리 저장 폴더: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_tainted-grail\postlaunch_preprocess_data
LLM 리뷰 결과: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_tainted-grail\llm_review_analysis_result.csv
LLM 이슈 태그 결과: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_tainted-grail\llm_issue_tags_flat.csv


# 2. 데이터 불러오기

In [3]:
# ============================================================
# 04번 LLM 분류 결과 불러오기
# ============================================================

result_df = pd.read_csv(RESULT_CSV_PATH, dtype={"recommendationid": "string"})
issue_df = pd.read_csv(ISSUE_TAG_FLAT_PATH, dtype={"recommendationid": "string"})

print("리뷰 단위 LLM 결과:", result_df.shape)
print("이슈 태그 단위 결과:", issue_df.shape)

display(result_df.head())
display(issue_df.head())


리뷰 단위 LLM 결과: (1000, 20)
이슈 태그 단위 결과: (1997, 15)


,analysis_status,recommendationid,appid,game_name,review_datetime,release_date,days_from_release,release_period,steam_label_text,playtime_at_review_hours,votes_up,weighted_vote_score,llm_sentiment,sentiment_score,llm_primary_issue,llm_issue_tags,llm_urgency_candidate,llm_review_summary,llm_suggested_action,steam_llm_sentiment_relation
0,success,221779395,1466060,Tainted Grail: The Fall of Avalon,2026-03-26 21:36:35,2025-05-23,307.0,D181+,positive,18.666667,0,0.5,positive,5,positive_praise,"[{""category"": ""positive_praise"", ""sentiment"": ""positive"", ""evidence"": ""Absolute banger of a game. Scratches both the...",low,엘더스크롤과 고딕 시리즈의 느낌을 잘 살린 훌륭한 게임이라고 평가함.,"현재의 게임 방향성을 유지하고, 커뮤니티의 긍정적인 피드백을 마케팅 자료로 활용할 것을 권장함.",exact_match
1,success,222596326,1466060,Tainted Grail: The Fall of Avalon,2026-04-05 16:40:12,2025-05-23,317.0,D181+,positive,27.050000,0,0.5,positive,5,gameplay_loop,"[{""category"": ""gameplay_loop"", ""sentiment"": ""positive"", ""evidence"": ""You can turn your hard-working summons... Into ...",low,소환수를 치즈로 변환하여 요리하거나 판매할 수 있는 독특한 게임 메커니즘을 매우 재미있게 즐기고 있음.,"사용자가 즐기는 독특한 게임 메커니즘(소환수 치즈화)이 의도된 것인지 확인하고, 향후 업데이트 시 이러한 창의적인 플레이 방식을 유지하거나 확장할지 검토할 것.",exact_match
2,success,219954559,1466060,Tainted Grail: The Fall of Avalon,2026-03-06 06:19:32,2025-05-23,287.0,D181+,positive,20.383333,1,0.5,positive,5,positive_praise,"[{""category"": ""positive_praise"", ""sentiment"": ""positive"", ""evidence"": ""Actually the best game ive played in years, l...",low,"최근 몇 년간 플레이한 게임 중 최고이며, 아트와 게임플레이 모두 훌륭하다고 평가함.","현재의 아트 스타일과 게임플레이 완성도를 유지하며, 향후 콘텐츠 업데이트 시에도 동일한 품질을 유지할 수 있도록 관리할 것.",exact_match
3,success,217220762,1466060,Tainted Grail: The Fall of Avalon,2026-01-31 03:40:11,2025-05-23,253.0,D181+,positive,52.250000,0,0.5,positive,5,positive_praise,"[{""category"": ""positive_praise"", ""sentiment"": ""positive"", ""evidence"": ""great game, great first game of this type for...",low,"개발사의 첫 시도치고 매우 훌륭한 게임이며, 지속적으로 플레이하고 응원하겠다는 긍정적인 리뷰입니다.","현재의 개발 방향성을 유지하고, 커뮤니티와의 소통을 통해 유저들의 피드백을 지속적으로 수렴하여 차기 업데이트에 반영하십시오.",exact_match
4,success,224253174,1466060,Tainted Grail: The Fall of Avalon,2026-04-28 01:50:32,2025-05-23,340.0,D181+,positive,9.533333,1,0.5,positive,5,positive_praise,"[{""category"": ""positive_praise"", ""sentiment"": ""positive"", ""evidence"": ""love-child of Skyrim, Souls, and the grim-dar...",low,"스카이림과 소울류 게임의 장점을 결합한 다크 판타지 게임으로, 버그가 적어 완성도가 높다는 긍정적인 평가입니다.","현재의 안정적인 빌드 상태를 유지하고, 향후 업데이트 시에도 버그 발생을 최소화하는 품질 관리 프로세스를 지속하십시오.",exact_match


,recommendationid,appid,game_name,steam_label_text,llm_sentiment,llm_primary_issue,llm_urgency_candidate,release_period,playtime_at_review_hours,votes_up,weighted_vote_score,llm_issue_category,issue_name_kor,llm_issue_sentiment,llm_issue_evidence
0,221779395,1466060,Tainted Grail: The Fall of Avalon,positive,positive,positive_praise,low,D181+,18.666667,0,0.5,positive_praise,긍정 칭찬,positive,Absolute banger of a game. Scratches both the Elder Scrolls as well as the Gothic/PB itch perfectly.
1,222596326,1466060,Tainted Grail: The Fall of Avalon,positive,positive,gameplay_loop,low,D181+,27.050000,0,0.5,gameplay_loop,게임플레이 루프,positive,"You can turn your hard-working summons... Into cheese... And then you can eat them, or sell them..."
2,219954559,1466060,Tainted Grail: The Fall of Avalon,positive,positive,positive_praise,low,D181+,20.383333,1,0.5,positive_praise,긍정 칭찬,positive,"Actually the best game ive played in years, love the art, love the gameplay"
3,217220762,1466060,Tainted Grail: The Fall of Avalon,positive,positive,positive_praise,low,D181+,52.250000,0,0.5,positive_praise,긍정 칭찬,positive,"great game, great first game of this type for these devs"
4,224253174,1466060,Tainted Grail: The Fall of Avalon,positive,positive,positive_praise,low,D181+,9.533333,1,0.5,positive_praise,긍정 칭찬,positive,"love-child of Skyrim, Souls, and the grim-dark aesthetic"


In [4]:
# ============================================================
# 필수 컬럼 확인
# ============================================================

required_result_cols = [
    "recommendationid", "appid", "game_name",
    "review_datetime", "release_date", "days_from_release", "release_period",
    "steam_label_text", "playtime_at_review_hours",
    "llm_sentiment", "llm_primary_issue", "llm_urgency_candidate",
    "llm_review_summary", "llm_suggested_action",
]

required_issue_cols = [
    "recommendationid", "appid", "game_name",
    "steam_label_text", "llm_sentiment", "llm_urgency_candidate",
    "release_period", "playtime_at_review_hours",
    "llm_issue_category", "issue_name_kor", "llm_issue_sentiment", "llm_issue_evidence",
]

missing_result_cols = [col for col in required_result_cols if col not in result_df.columns]
missing_issue_cols = [col for col in required_issue_cols if col not in issue_df.columns]

if missing_result_cols:
    raise ValueError(f"리뷰 결과 파일에 필요한 컬럼이 없습니다: {missing_result_cols}")

if missing_issue_cols:
    raise ValueError(f"이슈 태그 파일에 필요한 컬럼이 없습니다: {missing_issue_cols}")

print("필수 컬럼 확인 완료")


필수 컬럼 확인 완료


# 3. 리뷰 단위 전처리

리뷰 1개를 1행으로 두고, 이후 이슈 집계에 필요한 기본 플래그를 만든다.


In [5]:
# ============================================================
# 공통 함수
# ============================================================

def safe_rate(numerator, denominator):
    """0으로 나누는 경우를 방지한 비율 계산 함수."""
    if denominator is None or pd.isna(denominator) or denominator == 0:
        return 0.0
    return round(float(numerator) / float(denominator), 4)


def classify_playtime_stage(hours):
    """리뷰 작성 시점 플레이타임을 출시 후 운영 해석용 구간으로 분류한다."""
    if pd.isna(hours):
        return "unknown"
    if hours < 1:
        return "0-1h"
    if hours < 5:
        return "1-5h"
    if hours < 20:
        return "5-20h"
    if hours < 50:
        return "20-50h"
    return "50h+"


def classify_recency_group(review_dt, max_dt):
    """분석 데이터 내 최신 리뷰일 기준 최근성 구간을 만든다."""
    if pd.isna(review_dt) or pd.isna(max_dt):
        return "unknown"

    diff_days = (max_dt - review_dt).days

    if diff_days <= 30:
        return "last_30d"
    if diff_days <= 60:
        return "31-60d"
    if diff_days <= 90:
        return "61-90d"
    return "older_90d"


def normalize_text_value(x):
    """결측/공백 텍스트를 안정적으로 처리한다."""
    if pd.isna(x):
        return ""
    return str(x).strip()


In [6]:
# ============================================================
# 리뷰 단위 기본 전처리
# ============================================================

review_base = result_df.copy()

# 날짜/숫자 타입 정리
for col in ["review_datetime", "release_date"]:
    review_base[col] = pd.to_datetime(review_base[col], errors="coerce")

numeric_cols = [
    "days_from_release", "playtime_at_review_hours", "votes_up",
    "weighted_vote_score", "sentiment_score",
]
for col in numeric_cols:
    if col in review_base.columns:
        review_base[col] = pd.to_numeric(review_base[col], errors="coerce")

# 문자열 정리
text_cols = [
    "recommendationid", "game_name", "release_period", "steam_label_text",
    "llm_sentiment", "llm_primary_issue", "llm_urgency_candidate",
    "llm_review_summary", "llm_suggested_action",
]
for col in text_cols:
    if col in review_base.columns:
        review_base[col] = review_base[col].apply(normalize_text_value)

# 분석 기준일
MAX_REVIEW_DATETIME = review_base["review_datetime"].max()

# 파생 구간/플래그
review_base["playtime_stage"] = review_base["playtime_at_review_hours"].apply(classify_playtime_stage)
review_base["review_recency_group"] = review_base["review_datetime"].apply(lambda x: classify_recency_group(x, MAX_REVIEW_DATETIME))

review_base["steam_negative_flag"] = review_base["steam_label_text"].eq("negative")
review_base["steam_positive_flag"] = review_base["steam_label_text"].eq("positive")
review_base["llm_negative_or_mixed_flag"] = review_base["llm_sentiment"].isin(["negative", "mixed"])
review_base["llm_positive_flag"] = review_base["llm_sentiment"].eq("positive")
review_base["high_urgency_flag"] = review_base["llm_urgency_candidate"].eq("high")
review_base["early_playtime_flag"] = review_base["playtime_stage"].isin(["0-1h", "1-5h"])
review_base["recent_30d_flag"] = review_base["review_recency_group"].eq("last_30d")

# 저장 컬럼 정리
review_base_cols = [
    "analysis_status",
    "recommendationid", "appid", "game_name",
    "review_datetime", "release_date", "days_from_release", "release_period", "review_recency_group",
    "steam_label_text", "steam_positive_flag", "steam_negative_flag",
    "playtime_at_review_hours", "playtime_stage", "early_playtime_flag", "recent_30d_flag",
    "votes_up", "weighted_vote_score",
    "llm_sentiment", "sentiment_score", "llm_negative_or_mixed_flag", "llm_positive_flag",
    "llm_primary_issue", "llm_urgency_candidate", "high_urgency_flag",
    "llm_review_summary", "llm_suggested_action", "steam_llm_sentiment_relation",
]
review_base_cols = [col for col in review_base_cols if col in review_base.columns]
review_base = review_base[review_base_cols].copy()

print("리뷰 단위 전처리 결과:", review_base.shape)
print("최신 리뷰일:", MAX_REVIEW_DATETIME)
display(review_base.head())


리뷰 단위 전처리 결과: (1000, 28)
최신 리뷰일: 2026-04-29 01:42:34


,analysis_status,recommendationid,appid,game_name,review_datetime,release_date,days_from_release,release_period,review_recency_group,steam_label_text,steam_positive_flag,steam_negative_flag,playtime_at_review_hours,playtime_stage,early_playtime_flag,recent_30d_flag,votes_up,weighted_vote_score,llm_sentiment,sentiment_score,llm_negative_or_mixed_flag,llm_positive_flag,llm_primary_issue,llm_urgency_candidate,high_urgency_flag,llm_review_summary,llm_suggested_action,steam_llm_sentiment_relation
0,success,221779395,1466060,Tainted Grail: The Fall of Avalon,2026-03-26 21:36:35,2025-05-23,307.0,D181+,31-60d,positive,True,False,18.666667,5-20h,False,False,0,0.5,positive,5,False,True,positive_praise,low,False,엘더스크롤과 고딕 시리즈의 느낌을 잘 살린 훌륭한 게임이라고 평가함.,"현재의 게임 방향성을 유지하고, 커뮤니티의 긍정적인 피드백을 마케팅 자료로 활용할 것을 권장함.",exact_match
1,success,222596326,1466060,Tainted Grail: The Fall of Avalon,2026-04-05 16:40:12,2025-05-23,317.0,D181+,last_30d,positive,True,False,27.050000,20-50h,False,True,0,0.5,positive,5,False,True,gameplay_loop,low,False,소환수를 치즈로 변환하여 요리하거나 판매할 수 있는 독특한 게임 메커니즘을 매우 재미있게 즐기고 있음.,"사용자가 즐기는 독특한 게임 메커니즘(소환수 치즈화)이 의도된 것인지 확인하고, 향후 업데이트 시 이러한 창의적인 플레이 방식을 유지하거나 확장할지 검토할 것.",exact_match
2,success,219954559,1466060,Tainted Grail: The Fall of Avalon,2026-03-06 06:19:32,2025-05-23,287.0,D181+,31-60d,positive,True,False,20.383333,20-50h,False,False,1,0.5,positive,5,False,True,positive_praise,low,False,"최근 몇 년간 플레이한 게임 중 최고이며, 아트와 게임플레이 모두 훌륭하다고 평가함.","현재의 아트 스타일과 게임플레이 완성도를 유지하며, 향후 콘텐츠 업데이트 시에도 동일한 품질을 유지할 수 있도록 관리할 것.",exact_match
3,success,217220762,1466060,Tainted Grail: The Fall of Avalon,2026-01-31 03:40:11,2025-05-23,253.0,D181+,61-90d,positive,True,False,52.250000,50h+,False,False,0,0.5,positive,5,False,True,positive_praise,low,False,"개발사의 첫 시도치고 매우 훌륭한 게임이며, 지속적으로 플레이하고 응원하겠다는 긍정적인 리뷰입니다.","현재의 개발 방향성을 유지하고, 커뮤니티와의 소통을 통해 유저들의 피드백을 지속적으로 수렴하여 차기 업데이트에 반영하십시오.",exact_match
4,success,224253174,1466060,Tainted Grail: The Fall of Avalon,2026-04-28 01:50:32,2025-05-23,340.0,D181+,last_30d,positive,True,False,9.533333,5-20h,False,True,1,0.5,positive,5,False,True,positive_praise,low,False,"스카이림과 소울류 게임의 장점을 결합한 다크 판타지 게임으로, 버그가 적어 완성도가 높다는 긍정적인 평가입니다.","현재의 안정적인 빌드 상태를 유지하고, 향후 업데이트 시에도 버그 발생을 최소화하는 품질 관리 프로세스를 지속하십시오.",exact_match


# 4. 이슈 태그 단위 전처리

리뷰 안의 여러 이슈 태그를 펼친 데이터를 정리한다.  
한 리뷰가 같은 이슈를 여러 번 가진 경우, 이슈 집계에서는 `recommendationid + issue_category` 기준으로 중복을 제거한다.


In [7]:
# ============================================================
# 이슈 태그 단위 기본 전처리
# ============================================================

issue_base = issue_df.copy()

# 날짜/최근성/리뷰 요약 등은 리뷰 단위 결과에서 가져온다.
review_merge_cols = [
    "recommendationid", "review_datetime", "release_date", "days_from_release",
    "review_recency_group", "playtime_stage", "steam_negative_flag", "steam_positive_flag",
    "llm_negative_or_mixed_flag", "llm_positive_flag", "high_urgency_flag",
    "early_playtime_flag", "recent_30d_flag",
    "llm_review_summary", "llm_suggested_action", "steam_llm_sentiment_relation",
]
review_merge_cols = [col for col in review_merge_cols if col in review_base.columns]

issue_base = issue_base.merge(
    review_base[review_merge_cols].drop_duplicates("recommendationid"),
    on="recommendationid",
    how="left",
    suffixes=("", "_review"),
)

# 숫자 타입 정리
for col in ["playtime_at_review_hours", "votes_up", "weighted_vote_score", "days_from_release"]:
    if col in issue_base.columns:
        issue_base[col] = pd.to_numeric(issue_base[col], errors="coerce")

# 문자열 정리
for col in [
    "recommendationid", "game_name", "steam_label_text", "llm_sentiment",
    "llm_primary_issue", "llm_urgency_candidate", "release_period",
    "llm_issue_category", "issue_name_kor", "llm_issue_sentiment", "llm_issue_evidence",
]:
    if col in issue_base.columns:
        issue_base[col] = issue_base[col].apply(normalize_text_value)

# 이슈 태그 기준 감정 플래그
issue_base["issue_positive_flag"] = issue_base["llm_issue_sentiment"].eq("positive")
issue_base["issue_negative_or_mixed_flag"] = issue_base["llm_issue_sentiment"].isin(["negative", "mixed"])
issue_base["issue_negative_flag"] = issue_base["llm_issue_sentiment"].eq("negative")
issue_base["issue_mixed_flag"] = issue_base["llm_issue_sentiment"].eq("mixed")

# 같은 리뷰 안에서 같은 이슈가 중복으로 들어온 경우 제거
issue_review_base = issue_base.drop_duplicates(["recommendationid", "llm_issue_category"]).copy()

print("이슈 태그 원본 행 수:", len(issue_base))
print("리뷰-이슈 중복 제거 후 행 수:", len(issue_review_base))
display(issue_review_base.head())


이슈 태그 원본 행 수: 1997
리뷰-이슈 중복 제거 후 행 수: 1967


,recommendationid,appid,game_name,steam_label_text,llm_sentiment,llm_primary_issue,llm_urgency_candidate,release_period,playtime_at_review_hours,votes_up,weighted_vote_score,llm_issue_category,issue_name_kor,llm_issue_sentiment,llm_issue_evidence,review_datetime,release_date,days_from_release,review_recency_group,playtime_stage,steam_negative_flag,steam_positive_flag,llm_negative_or_mixed_flag,llm_positive_flag,high_urgency_flag,early_playtime_flag,recent_30d_flag,llm_review_summary,llm_suggested_action,steam_llm_sentiment_relation,issue_positive_flag,issue_negative_or_mixed_flag,issue_negative_flag,issue_mixed_flag
0,221779395,1466060,Tainted Grail: The Fall of Avalon,positive,positive,positive_praise,low,D181+,18.666667,0,0.5,positive_praise,긍정 칭찬,positive,Absolute banger of a game. Scratches both the Elder Scrolls as well as the Gothic/PB itch perfectly.,2026-03-26 21:36:35,2025-05-23,307.0,31-60d,5-20h,False,True,False,True,False,False,False,엘더스크롤과 고딕 시리즈의 느낌을 잘 살린 훌륭한 게임이라고 평가함.,"현재의 게임 방향성을 유지하고, 커뮤니티의 긍정적인 피드백을 마케팅 자료로 활용할 것을 권장함.",exact_match,True,False,False,False
1,222596326,1466060,Tainted Grail: The Fall of Avalon,positive,positive,gameplay_loop,low,D181+,27.050000,0,0.5,gameplay_loop,게임플레이 루프,positive,"You can turn your hard-working summons... Into cheese... And then you can eat them, or sell them...",2026-04-05 16:40:12,2025-05-23,317.0,last_30d,20-50h,False,True,False,True,False,False,True,소환수를 치즈로 변환하여 요리하거나 판매할 수 있는 독특한 게임 메커니즘을 매우 재미있게 즐기고 있음.,"사용자가 즐기는 독특한 게임 메커니즘(소환수 치즈화)이 의도된 것인지 확인하고, 향후 업데이트 시 이러한 창의적인 플레이 방식을 유지하거나 확장할지 검토할 것.",exact_match,True,False,False,False
2,219954559,1466060,Tainted Grail: The Fall of Avalon,positive,positive,positive_praise,low,D181+,20.383333,1,0.5,positive_praise,긍정 칭찬,positive,"Actually the best game ive played in years, love the art, love the gameplay",2026-03-06 06:19:32,2025-05-23,287.0,31-60d,20-50h,False,True,False,True,False,False,False,"최근 몇 년간 플레이한 게임 중 최고이며, 아트와 게임플레이 모두 훌륭하다고 평가함.","현재의 아트 스타일과 게임플레이 완성도를 유지하며, 향후 콘텐츠 업데이트 시에도 동일한 품질을 유지할 수 있도록 관리할 것.",exact_match,True,False,False,False
3,217220762,1466060,Tainted Grail: The Fall of Avalon,positive,positive,positive_praise,low,D181+,52.250000,0,0.5,positive_praise,긍정 칭찬,positive,"great game, great first game of this type for these devs",2026-01-31 03:40:11,2025-05-23,253.0,61-90d,50h+,False,True,False,True,False,False,False,"개발사의 첫 시도치고 매우 훌륭한 게임이며, 지속적으로 플레이하고 응원하겠다는 긍정적인 리뷰입니다.","현재의 개발 방향성을 유지하고, 커뮤니티와의 소통을 통해 유저들의 피드백을 지속적으로 수렴하여 차기 업데이트에 반영하십시오.",exact_match,True,False,False,False
4,224253174,1466060,Tainted Grail: The Fall of Avalon,positive,positive,positive_praise,low,D181+,9.533333,1,0.5,positive_praise,긍정 칭찬,positive,"love-child of Skyrim, Souls, and the grim-dark aesthetic",2026-04-28 01:50:32,2025-05-23,340.0,last_30d,5-20h,False,True,False,True,False,False,True,"스카이림과 소울류 게임의 장점을 결합한 다크 판타지 게임으로, 버그가 적어 완성도가 높다는 긍정적인 평가입니다.","현재의 안정적인 빌드 상태를 유지하고, 향후 업데이트 시에도 버그 발생을 최소화하는 품질 관리 프로세스를 지속하십시오.",exact_match,True,False,False,False


# 5. 게임 단위 기본 요약

분석 대상 게임의 현재 상태를 확인하기 위한 내부 요약값을 만든다.  
다만 이번 04-1 산출물에서는 별도 CSV로 저장하지 않고, 필요하면 노트북 화면에서만 확인한다.

In [8]:
# ============================================================
# 게임 단위 기본 요약
# ============================================================

def make_game_base(review_df, issue_review_df):
    rows = []

    for (appid, game_name), g in review_df.groupby(["appid", "game_name"], dropna=False):
        issue_g = issue_review_df[issue_review_df["appid"] == appid]

        review_count = g["recommendationid"].nunique()
        steam_positive_count = int(g["steam_positive_flag"].sum())
        steam_negative_count = int(g["steam_negative_flag"].sum())
        llm_positive_count = int(g["llm_positive_flag"].sum())
        llm_negative_mixed_count = int(g["llm_negative_or_mixed_flag"].sum())
        high_urgency_count = int(g["high_urgency_flag"].sum())

        rows.append({
            "appid": appid,
            "game_name": game_name,
            "review_count": review_count,
            "issue_tag_count": len(issue_g),
            "steam_positive_review_count": steam_positive_count,
            "steam_negative_review_count": steam_negative_count,
            "steam_positive_rate": safe_rate(steam_positive_count, review_count),
            "llm_positive_review_count": llm_positive_count,
            "llm_negative_mixed_review_count": llm_negative_mixed_count,
            "llm_negative_mixed_rate": safe_rate(llm_negative_mixed_count, review_count),
            "high_urgency_review_count": high_urgency_count,
            "high_urgency_rate": safe_rate(high_urgency_count, review_count),
            "first_review_datetime": g["review_datetime"].min(),
            "last_review_datetime": g["review_datetime"].max(),
            "mean_playtime_at_review_hours": round(g["playtime_at_review_hours"].mean(), 2),
            "median_playtime_at_review_hours": round(g["playtime_at_review_hours"].median(), 2),
        })

    return pd.DataFrame(rows)


game_base = make_game_base(review_base, issue_review_base)

print("게임 단위 요약:", game_base.shape)
display(game_base)


게임 단위 요약: (1, 16)


,appid,game_name,review_count,issue_tag_count,steam_positive_review_count,steam_negative_review_count,steam_positive_rate,llm_positive_review_count,llm_negative_mixed_review_count,llm_negative_mixed_rate,high_urgency_review_count,high_urgency_rate,first_review_datetime,last_review_datetime,mean_playtime_at_review_hours,median_playtime_at_review_hours
0,1466060,Tainted Grail: The Fall of Avalon,1000,1967,671,329,0.671,592,405,0.405,178,0.178,2026-01-29 00:40:27,2026-04-29 01:42:34,41.08,30.22


# 6. 이슈 단위 요약

패치·운영 전략 생성의 핵심이 되는 이슈별 반복성 근거를 만든다.

여기서의 `action_group_hint`, `rule_priority_hint`는 LLM 판단값이 아니라 **04-1에서 사전에 정한 데이터 기준으로 계산한 파생 컬럼**이다.

## 우선순위 계산 원칙

| 기준 | 의미 | 반영 방식 |
|---|---|---|
| 이슈 반복 수 | 같은 이슈가 몇 개 리뷰에서 반복되는지 | 핵심 기준 |
| 부정·혼합 맥락 | LLM이 해당 이슈를 부정/혼합 맥락으로 분류했는지 | 핵심 기준 |
| Steam 비추천 맥락 | Steam 라벨 기준 비추천 리뷰에서도 해당 이슈가 나타나는지 | 핵심 기준 |
| 최근성 | 최근 30일에도 같은 문제가 반복되는지 | 현재 운영 판단 기준 |
| 짧은 플레이타임 | 초반 플레이타임에서 부정·혼합 이슈가 나타나는지 | 초기 이탈 가능성 참고 |
| High urgency | 04번 LLM 리뷰 분류에서 나온 시급도 후보 | **우선순위 계산에는 직접 사용하지 않고 보조 설명으로만 사용** |

따라서 `High urgency`가 높다는 이유만으로 `상` 우선순위를 부여하지 않는다.  
`상` 우선순위는 반복성, 부정·혼합 맥락, Steam 비추천 맥락, 최근성 중 핵심 근거가 함께 확인될 때만 부여한다.


In [ ]:
# ============================================================
# 대응 구분/우선 검토 힌트 함수
# ============================================================
# 튜터님 피드백 반영:
# 패치·운영 우선순위는 LLM이 직접 판단하지 않는다.
# 이 셀에서 사전에 정한 데이터 기준으로 action_group_hint와 rule_priority_hint를 계산한다.
#
# 핵심 기준:
# 1. 해당 이슈가 리뷰에서 반복적으로 나타났는가
# 2. 부정·혼합 맥락으로 자주 언급되었는가
# 3. Steam 비추천 리뷰에서도 함께 나타났는가
# 4. 최근 30일에도 반복되는가
# 5. 짧은 플레이타임 구간에서도 부정·혼합으로 나타나는가
#
# 보조 지표:
# - high_urgency_review_count는 04번 LLM 리뷰 분류에서 나온 시급도 후보의 집계값이다.
# - 따라서 rule_priority_hint 계산에는 직접 사용하지 않는다.
# - priority_reason에서만 "보조 참고 지표"로 남긴다.

# ------------------------------------------------------------
# 실제 04번 LLM 결과의 llm_issue_category 값과 맞춘 이슈 그룹
# ------------------------------------------------------------
IMMEDIATE_ISSUES = {
    "crash",
    "save_progression",
    "save_progress",
    "bug",
    "performance",
    "optimization",
}

SHORT_TERM_ISSUES = {
    "gameplay_loop",
    "balance",
    "difficulty",
    "ui_ux",
    "control",
    "controls",
    "progression_grind",
}

LONG_TERM_ISSUES = {
    "content_volume",
    "content_amount",
    "story",
    "graphics_audio",
    "multiplayer",
    "monetization",
    "pricing",
    "price_value",
    "translation_localization",
}

STRENGTH_ISSUES = {"positive_praise"}
OPS_ISSUES = {"developer_communication"}

# ------------------------------------------------------------
# 우선순위 기준
# ------------------------------------------------------------
# 상:
# - 부정·혼합 맥락이 충분히 반복되고
# - Steam 비추천 맥락도 함께 확인되며
# - 최근 30일에도 반복되거나, 전체 반복 규모가 큰 이슈
#
# 중:
# - 부정·혼합 또는 Steam 비추천 맥락이 일정 수준 확인되는 이슈
#
# 하:
# - 반복성과 부정 맥락 근거가 약하거나, 강점 유지/참고 이슈

HIGH_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS = 20
HIGH_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS = 10
HIGH_PRIORITY_MIN_RECENT_NEGATIVE_MIXED_REVIEWS = 5
HIGH_PRIORITY_MIN_AFFECTED_REVIEWS = 50

MID_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS = 10
MID_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS = 5
MID_PRIORITY_MIN_RECENT_NEGATIVE_MIXED_REVIEWS = 3
MID_PRIORITY_MIN_AFFECTED_REVIEWS = 20

PATCH_OPS_NOTE_MAP = {
    "crash": "크래시 발생 조건과 로그를 우선 확인하고, 재현 가능한 오류부터 수정한다.",
    "save_progression": "저장 손실, 진행 막힘, 퀘스트 진행 불가 여부를 우선 점검한다.",
    "save_progress": "저장 손실, 진행 막힘, 퀘스트 진행 불가 여부를 우선 점검한다.",
    "bug": "반복 언급된 버그를 재현 가능성 기준으로 분류하고, 플레이 방해 수준이 큰 항목부터 수정한다.",
    "performance": "프레임 저하, 로딩, 끊김 등 성능 문제를 환경별로 점검한다.",
    "optimization": "최적화 이슈가 특정 구간이나 사양에서 반복되는지 확인한다.",
    "gameplay_loop": "반복 피로, 목표 구조, 보상 흐름을 점검하고 플레이 루프의 지루함을 줄인다.",
    "balance": "전투, 성장, 보상, 적 난이도의 불균형 지점을 조정한다.",
    "difficulty": "초반 진입 장벽과 후반 난이도 피로를 구분해 난이도 옵션 또는 안내를 보강한다.",
    "ui_ux": "메뉴, 인벤토리, 퀘스트 안내, 조작 안내 등 편의성 문제를 개선한다.",
    "control": "이동, 전투, 상호작용 조작의 반응성과 키 설정 편의성을 점검한다.",
    "controls": "이동, 전투, 상호작용 조작의 반응성과 키 설정 편의성을 점검한다.",
    "progression_grind": "반복 성장과 노가다 피로를 줄일 수 있는 보상/성장 속도 조정을 검토한다.",
    "content_volume": "콘텐츠 부족·반복성은 단기 패치보다 업데이트 로드맵 관점에서 검토한다.",
    "content_amount": "콘텐츠 부족·반복성은 단기 패치보다 업데이트 로드맵 관점에서 검토한다.",
    "story": "서사 전달, 퀘스트 흐름, 엔딩/분기 만족도를 장기 개선 후보로 검토한다.",
    "graphics_audio": "그래픽/사운드가 몰입을 방해하는지와 강점으로 작동하는지를 함께 확인한다.",
    "multiplayer": "멀티플레이 요구가 반복되는 경우 개발 범위와 수요를 장기 로드맵에서 검토한다.",
    "monetization": "가격, DLC, 과금 관련 불만이 반복되는지 확인하고 커뮤니케이션 방식을 점검한다.",
    "pricing": "가격 대비 만족도 불만이 반복되는지 확인하고 할인/번들/콘텐츠 가치 전달을 검토한다.",
    "price_value": "가격 대비 만족도 불만이 반복되는지 확인하고 할인/번들/콘텐츠 가치 전달을 검토한다.",
    "translation_localization": "번역/현지화 불만이 실제 이해도와 진행 경험에 영향을 주는지 검토한다.",
    "developer_communication": "패치 노트, 공지, 커뮤니티 응답 등 운영 커뮤니케이션을 점검한다.",
    "positive_praise": "긍정적으로 평가된 요소를 유지하고, 업데이트와 마케팅 메시지에서 강점으로 활용한다.",
    "other": "세부 리뷰를 확인해 반복되는 하위 원인이 있는지 검토한다.",
}


def get_action_group_hint(row):
    """이슈 성격과 부정 맥락을 기준으로 대응 구분을 계산한다."""
    issue = row["llm_issue_category"]
    affected = int(row["affected_review_count"])
    positive = int(row["positive_review_count"])
    neg_mixed = int(row["negative_mixed_review_count"])
    steam_negative = int(row["steam_negative_review_count"])

    # 긍정 칭찬은 개선 우선순위가 아니라 유지할 강점으로 분리한다.
    if issue in STRENGTH_ISSUES and positive >= neg_mixed:
        return "강점 유지"

    # 즉시 확인은 크래시/저장/버그/성능/최적화처럼 플레이를 직접 방해할 수 있는 이슈로 제한한다.
    # High urgency만으로 즉시 확인으로 올리지 않는다.
    if issue in IMMEDIATE_ISSUES and neg_mixed >= 5 and steam_negative >= 3:
        return "즉시 확인"

    if issue in SHORT_TERM_ISSUES and neg_mixed >= 5:
        return "단기 개선"

    if issue in OPS_ISSUES and (neg_mixed >= 3 or steam_negative >= 3):
        return "운영 커뮤니케이션 개선"

    if issue in LONG_TERM_ISSUES and neg_mixed >= 5:
        return "장기 검토"

    if positive > neg_mixed and positive >= max(10, affected * 0.5):
        return "강점 유지"

    return "검토 필요"


def get_rule_priority_hint(row):
    """규칙 기반 우선 검토 수준을 계산한다.

    High urgency는 LLM 기반 보조 지표이므로 이 함수의 계산식에 넣지 않는다.
    """
    action_group = row["action_group_hint"]
    affected = int(row["affected_review_count"])
    neg_mixed = int(row["negative_mixed_review_count"])
    steam_negative = int(row["steam_negative_review_count"])
    recent_negative = int(row["recent_30d_negative_mixed_review_count"])

    if action_group == "강점 유지":
        return "하"

    # 상: 부정·혼합 반복 + Steam 비추천 맥락 + 최근성 또는 큰 반복 규모가 함께 확인되는 경우
    high_by_recent = (
        neg_mixed >= HIGH_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS
        and steam_negative >= HIGH_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS
        and recent_negative >= HIGH_PRIORITY_MIN_RECENT_NEGATIVE_MIXED_REVIEWS
    )

    high_by_volume = (
        affected >= HIGH_PRIORITY_MIN_AFFECTED_REVIEWS
        and neg_mixed >= HIGH_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS
        and steam_negative >= HIGH_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS
    )

    # 즉시 확인 그룹은 기술/진행 차단 가능성이 있으므로 최근성 기준이 조금 약해도 상으로 둔다.
    high_by_blocking_issue = (
        action_group == "즉시 확인"
        and neg_mixed >= 15
        and steam_negative >= 10
    )

    if high_by_recent or high_by_volume or high_by_blocking_issue:
        return "상"

    # 중: 부정·혼합 또는 Steam 비추천 맥락이 일정 수준 확인되는 경우
    mid_by_negative = (
        neg_mixed >= MID_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS
        and steam_negative >= MID_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS
    )

    mid_by_recent = (
    recent_negative >= MID_PRIORITY_MIN_RECENT_NEGATIVE_MIXED_REVIEWS
    and steam_negative >= 1
    )

    mid_by_volume = (
        affected >= MID_PRIORITY_MIN_AFFECTED_REVIEWS
        and neg_mixed >= 5
    )

    if mid_by_negative or mid_by_recent or mid_by_volume:
        return "중"

    return "하"


def get_priority_rule_detail(row):
    """우선순위가 어떤 규칙 때문에 부여되었는지 설명용 라벨을 만든다."""
    priority = row["rule_priority_hint"]
    action_group = row["action_group_hint"]
    affected = int(row["affected_review_count"])
    neg_mixed = int(row["negative_mixed_review_count"])
    steam_negative = int(row["steam_negative_review_count"])
    recent_negative = int(row["recent_30d_negative_mixed_review_count"])

    if action_group == "강점 유지":
        return "강점 유지 항목이므로 개선 우선순위 산정 대상에서 분리"

    if priority == "상":
        if action_group == "즉시 확인" and neg_mixed >= 15 and steam_negative >= 10:
            return "플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락"
        if recent_negative >= HIGH_PRIORITY_MIN_RECENT_NEGATIVE_MIXED_REVIEWS:
            return "부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복"
        if affected >= HIGH_PRIORITY_MIN_AFFECTED_REVIEWS:
            return "전체 반복 규모 큼 + 부정·혼합 반복 + Steam 비추천 맥락"
        return "상 우선순위 규칙 충족"

    if priority == "중":
        if neg_mixed >= MID_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS and steam_negative >= MID_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS:
            return "부정·혼합 맥락과 Steam 비추천 맥락이 일정 수준 확인"
        if recent_negative >= MID_PRIORITY_MIN_RECENT_NEGATIVE_MIXED_REVIEWS:
            return "최근 30일 부정·혼합 반복이 일부 확인"
        if affected >= MID_PRIORITY_MIN_AFFECTED_REVIEWS:
            return "전체 반복 규모는 있으나 상 기준에는 미달"
        return "중 우선순위 규칙 충족"

    return "반복성 또는 Steam 비추천 맥락 근거가 상대적으로 약함"


def build_priority_reason(row):
    """표와 LLM 입력에 넣을 근거 문장을 만든다."""
    affected = int(row["affected_review_count"])
    neg_mixed = int(row["negative_mixed_review_count"])
    steam_negative = int(row["steam_negative_review_count"])
    recent_negative = int(row["recent_30d_negative_mixed_review_count"])
    early_negative = int(row["early_playtime_negative_mixed_review_count"])
    high = int(row["high_urgency_review_count"])

    parts = [
        f"영향 리뷰 {affected}개",
        f"부정·혼합 {neg_mixed}개",
        f"Steam 비추천 맥락 {steam_negative}개",
    ]

    if recent_negative > 0:
        parts.append(f"최근 30일 부정·혼합 {recent_negative}개")

    if early_negative > 0:
        parts.append(f"초반 플레이타임 부정·혼합 {early_negative}개")

    # High urgency는 우선순위 계산값이 아니라 보조 참고 지표로만 적는다.
    if high > 0:
        parts.append(f"High urgency 후보 {high}개(보조 참고)")

    parts.append(f"규칙 근거: {row['priority_rule_detail']}")
    return ", ".join(parts)


In [10]:
# ============================================================
# 이슈 단위 요약 생성
# ============================================================

def summarize_issue_group(g):
    affected = g["recommendationid"].nunique()

    positive_ids = g.loc[g["issue_positive_flag"], "recommendationid"].nunique()
    negative_ids = g.loc[g["issue_negative_flag"], "recommendationid"].nunique()
    mixed_ids = g.loc[g["issue_mixed_flag"], "recommendationid"].nunique()
    negative_mixed_ids = g.loc[g["issue_negative_or_mixed_flag"], "recommendationid"].nunique()

    high_ids = g.loc[g["high_urgency_flag"], "recommendationid"].nunique()
    high_negative_mixed_ids = g.loc[
        g["high_urgency_flag"] & g["issue_negative_or_mixed_flag"],
        "recommendationid"
    ].nunique()

    steam_negative_ids = g.loc[g["steam_negative_flag"], "recommendationid"].nunique()

    recent_30_ids = g.loc[g["recent_30d_flag"], "recommendationid"].nunique()
    recent_30_neg_mixed_ids = g.loc[
        g["recent_30d_flag"] & g["issue_negative_or_mixed_flag"],
        "recommendationid"
    ].nunique()

    early_neg_mixed_ids = g.loc[
        g["early_playtime_flag"] & g["issue_negative_or_mixed_flag"],
        "recommendationid"
    ].nunique()

    return pd.Series({
        "appid": g["appid"].iloc[0],
        "game_name": g["game_name"].iloc[0],
        "issue_name_kor": g["issue_name_kor"].iloc[0],
        "affected_review_count": affected,
        "positive_review_count": positive_ids,
        "negative_review_count": negative_ids,
        "mixed_review_count": mixed_ids,
        "negative_mixed_review_count": negative_mixed_ids,
        "steam_negative_review_count": steam_negative_ids,
        "high_urgency_review_count": high_ids,
        "high_urgency_negative_mixed_review_count": high_negative_mixed_ids,
        "recent_30d_review_count": recent_30_ids,
        "recent_30d_negative_mixed_review_count": recent_30_neg_mixed_ids,
        "early_playtime_negative_mixed_review_count": early_neg_mixed_ids,
        "avg_playtime_at_review_hours": round(g["playtime_at_review_hours"].mean(), 2),
        "median_playtime_at_review_hours": round(g["playtime_at_review_hours"].median(), 2),
    })


issue_summary = (
    issue_review_base
    .groupby("llm_issue_category", dropna=False)
    .apply(summarize_issue_group, include_groups=False)
    .reset_index()
)

issue_summary["high_urgency_rate"] = issue_summary.apply(
    lambda row: safe_rate(row["high_urgency_review_count"], row["affected_review_count"]),
    axis=1,
)
issue_summary["negative_mixed_rate"] = issue_summary.apply(
    lambda row: safe_rate(row["negative_mixed_review_count"], row["affected_review_count"]),
    axis=1,
)
issue_summary["steam_negative_rate"] = issue_summary.apply(
    lambda row: safe_rate(row["steam_negative_review_count"], row["affected_review_count"]),
    axis=1,
)
issue_summary["recent_30d_negative_mixed_rate"] = issue_summary.apply(
    lambda row: safe_rate(row["recent_30d_negative_mixed_review_count"], row["recent_30d_review_count"]),
    axis=1,
)

issue_summary["action_group_hint"] = issue_summary.apply(get_action_group_hint, axis=1)
issue_summary["rule_priority_hint"] = issue_summary.apply(get_rule_priority_hint, axis=1)
issue_summary["priority_rule_detail"] = issue_summary.apply(get_priority_rule_detail, axis=1)
issue_summary["priority_reason"] = issue_summary.apply(build_priority_reason, axis=1)
issue_summary["patch_ops_note"] = issue_summary["llm_issue_category"].map(PATCH_OPS_NOTE_MAP).fillna("세부 리뷰 확인 후 대응 방향을 검토한다.")

# 보고서에서 보기 좋은 정렬
priority_order = {"상": 0, "중": 1, "하": 2}
action_order = {"즉시 확인": 0, "단기 개선": 1, "운영 커뮤니케이션 개선": 2, "장기 검토": 3, "검토 필요": 4, "강점 유지": 5}

issue_summary["priority_order"] = issue_summary["rule_priority_hint"].map(priority_order).fillna(9)
issue_summary["action_order"] = issue_summary["action_group_hint"].map(action_order).fillna(9)

issue_summary = issue_summary.sort_values(
    [
        "priority_order",
        "action_order",
        "negative_mixed_review_count",
        "steam_negative_review_count",
        "recent_30d_negative_mixed_review_count",
        "affected_review_count",
    ],
    ascending=[True, True, False, False, False, False],
).drop(columns=["priority_order", "action_order"])

print("이슈 단위 요약:", issue_summary.shape)
display(issue_summary.head(20))


이슈 단위 요약: (21, 26)


,llm_issue_category,appid,game_name,issue_name_kor,affected_review_count,positive_review_count,negative_review_count,mixed_review_count,negative_mixed_review_count,steam_negative_review_count,high_urgency_review_count,high_urgency_negative_mixed_review_count,recent_30d_review_count,recent_30d_negative_mixed_review_count,early_playtime_negative_mixed_review_count,avg_playtime_at_review_hours,median_playtime_at_review_hours,high_urgency_rate,negative_mixed_rate,steam_negative_rate,recent_30d_negative_mixed_rate,action_group_hint,rule_priority_hint,priority_rule_detail,priority_reason,patch_ops_note
1,bug,1466060,Tainted Grail: The Fall of Avalon,버그,95,4,85,0,85,41,38,38,38,33,11,44.86,34.28,0.4000,0.8947,0.4316,0.8684,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,"영향 리뷰 95개, 부정·혼합 85개, Steam 비추천 맥락 41개, 최근 30일 부정·혼합 33개, 초반 플레이타임 부정·혼합 11개, High urgency 후보 38개(보조 참고), 규칙 근거: 플레이...","반복 언급된 버그를 재현 가능성 기준으로 분류하고, 플레이 방해 수준이 큰 항목부터 수정한다."
13,performance,1466060,Tainted Grail: The Fall of Avalon,성능,60,5,53,0,53,26,26,26,25,22,13,26.13,19.30,0.4333,0.8833,0.4333,0.8800,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,"영향 리뷰 60개, 부정·혼합 53개, Steam 비추천 맥락 26개, 최근 30일 부정·혼합 22개, 초반 플레이타임 부정·혼합 13개, High urgency 후보 26개(보조 참고), 규칙 근거: 플레이...","프레임 저하, 로딩, 끊김 등 성능 문제를 환경별로 점검한다."
4,crash,1466060,Tainted Grail: The Fall of Avalon,크래시,30,0,29,0,29,25,26,26,12,11,7,28.43,20.27,0.8667,0.9667,0.8333,0.9167,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,"영향 리뷰 30개, 부정·혼합 29개, Steam 비추천 맥락 25개, 최근 30일 부정·혼합 11개, 초반 플레이타임 부정·혼합 7개, High urgency 후보 26개(보조 참고), 규칙 근거: 플레이 ...","크래시 발생 조건과 로그를 우선 확인하고, 재현 가능한 오류부터 수정한다."
11,optimization,1466060,Tainted Grail: The Fall of Avalon,최적화,27,1,25,1,26,16,14,14,11,11,11,19.36,8.40,0.5185,0.9630,0.5926,1.0000,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,"영향 리뷰 27개, 부정·혼합 26개, Steam 비추천 맥락 16개, 최근 30일 부정·혼합 11개, 초반 플레이타임 부정·혼합 11개, High urgency 후보 14개(보조 참고), 규칙 근거: 플레이...",최적화 이슈가 특정 구간이나 사양에서 반복되는지 확인한다.
17,save_progression,1466060,Tainted Grail: The Fall of Avalon,저장/진행,27,1,24,0,24,23,23,23,8,8,3,26.12,20.45,0.8519,0.8889,0.8519,1.0000,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,"영향 리뷰 27개, 부정·혼합 24개, Steam 비추천 맥락 23개, 최근 30일 부정·혼합 8개, 초반 플레이타임 부정·혼합 3개, High urgency 후보 23개(보조 참고), 규칙 근거: 플레이 방...","저장 손실, 진행 막힘, 퀘스트 진행 불가 여부를 우선 점검한다."
7,gameplay_loop,1466060,Tainted Grail: The Fall of Avalon,게임플레이 루프,284,58,202,6,208,150,73,71,112,81,41,35.87,24.38,0.2570,0.7324,0.5282,0.7232,단기 개선,상,부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복,"영향 리뷰 284개, 부정·혼합 208개, Steam 비추천 맥락 150개, 최근 30일 부정·혼합 81개, 초반 플레이타임 부정·혼합 41개, High urgency 후보 73개(보조 참고), 규칙 근거: ...","반복 피로, 목표 구조, 보상 흐름을 점검하고 플레이 루프의 지루함을 줄인다."
0,balance,1466060,Tainted Grail: The Fall of Avalon,밸런스,130,3,112,6,118,64,40,40,55,50,6,47.97,34.20,0.3077,0.9077,0.4923,0.9091,단기 개선,상,부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복,"영향 리뷰 130개, 부정·혼합 118개, Steam 비추천 맥락 64개, 최근 30일 부정·혼합 50개, 초반 플레이타임 부정·혼합 6개, High urgency 후보 40개(보조 참고), 규칙 근거: 부정...","전투, 성장, 보상, 적 난이도의 불균형 지점을 조정한다."
20,ui_ux,1466060,Tainted Grail: The Fall of Avalon,UI/UX,63,2,58,0,58,36,20,20,22,19,8,41.18,25.98,0.3175,0.9206,0.5714,0.8636,단기 개선,상,부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복,"영향 리뷰 63개, 부정·혼합 58개, Steam 비추천 맥락 36개, 최근 30일 부정·혼합 19개, 초반 플레이타임 부정·혼합 8개, High urgency 후보 20개(보조 참고), 규칙 근거: 부정·혼...","메뉴, 인벤토리, 퀘스트 안내, 조작 안내 등 편의성 문제를 개선한다."
6,difficulty,1466060,Tainted Grail: The Fall of Avalon,난이도,68,9,48,0,48,30,19,19,26,21,7,42.24,25.36,0.2794,0.7059,0.4412,0.8077,단기 개선,상,부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복,"영향 리뷰 68개, 부정·혼합 48개, Steam 비추천 맥락 30개, 최근 30일 부정·혼합 21개, 초반 플레이타임 부정·혼합 7개, High urgency 후보 19개(보조 참고), 규칙 근거: 부정·혼...",초반 진입 장벽과 후반 난이도 피로를 구분해 난이도 옵션 또는 안내를 보강한다.
16,progression_grind,1466060,Tainted Grail: The Fall of Avalon,성장/반복 노가다,28,1,26,0,26,16,14,14,9,9,2,57.39,31.98,0.5000,0.9286,0.5714,1.0000,단기 개선,상,부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복,"영향 리뷰 28개, 부정·혼합 26개, Steam 비추천 맥락 16개, 최근 30일 부정·혼합 9개, 초반 플레이타임 부정·혼합 2개, High urgency 후보 14개(보조 참고), 규칙 근거: 부정·혼합...",반복 성장과 노가다 피로를 줄일 수 있는 보상/성장 속

# 7. 플레이타임/최근성 기준 이슈 요약

패치·운영 전략에서 다음 질문에 답하기 위한 보조 집계다.

- 초반 플레이에서 불만이 많은 이슈는 무엇인가?
- 최근 30일에도 반복되는 이슈는 무엇인가?

이 결과는 `postlaunch_issue_summary.csv`와 `postlaunch_patch_ops_evidence_base.csv`에 필요한 값으로 반영하고, 별도 CSV로는 저장하지 않는다.

In [11]:
# ============================================================
# 플레이타임 구간별 이슈 요약
# ============================================================

playtime_order = ["0-1h", "1-5h", "5-20h", "20-50h", "50h+", "unknown"]

playtime_issue_summary = (
    issue_review_base
    .groupby(["playtime_stage", "llm_issue_category", "issue_name_kor"], dropna=False)
    .agg(
        affected_review_count=("recommendationid", "nunique"),
        negative_mixed_review_count=("issue_negative_or_mixed_flag", "sum"),
        high_urgency_review_count=("high_urgency_flag", "sum"),
        steam_negative_review_count=("steam_negative_flag", "sum"),
    )
    .reset_index()
)

playtime_issue_summary["playtime_stage"] = pd.Categorical(
    playtime_issue_summary["playtime_stage"],
    categories=playtime_order,
    ordered=True,
)
playtime_issue_summary = playtime_issue_summary.sort_values(
    ["playtime_stage", "negative_mixed_review_count", "high_urgency_review_count"],
    ascending=[True, False, False],
)

print("플레이타임 구간별 이슈 요약:", playtime_issue_summary.shape)
display(playtime_issue_summary.head(20))


플레이타임 구간별 이슈 요약: (92, 7)


,playtime_stage,llm_issue_category,issue_name_kor,affected_review_count,negative_mixed_review_count,high_urgency_review_count,steam_negative_review_count
6,0-1h,gameplay_loop,게임플레이 루프,13,12,4,12
7,0-1h,graphics_audio,그래픽/사운드,10,7,4,9
9,0-1h,other,기타,6,6,1,6
8,0-1h,optimization,최적화,4,4,3,4
15,0-1h,ui_ux,UI/UX,4,4,3,4
5,0-1h,difficulty,난이도,4,4,2,4
10,0-1h,performance,성능,3,3,3,3
14,0-1h,story,스토리,3,3,0,3
3,0-1h,crash,크래시,2,2,2,2
4,0-1h,developer_communication,개발사 소통,2,2,2,2


In [12]:
# ============================================================
# 최근성 구간별 이슈 요약
# ============================================================

recency_order = ["last_30d", "31-60d", "61-90d", "older_90d", "unknown"]

recency_issue_summary = (
    issue_review_base
    .groupby(["review_recency_group", "llm_issue_category", "issue_name_kor"], dropna=False)
    .agg(
        affected_review_count=("recommendationid", "nunique"),
        negative_mixed_review_count=("issue_negative_or_mixed_flag", "sum"),
        high_urgency_review_count=("high_urgency_flag", "sum"),
        steam_negative_review_count=("steam_negative_flag", "sum"),
    )
    .reset_index()
)

recency_issue_summary["review_recency_group"] = pd.Categorical(
    recency_issue_summary["review_recency_group"],
    categories=recency_order,
    ordered=True,
)
recency_issue_summary = recency_issue_summary.sort_values(
    ["review_recency_group", "negative_mixed_review_count", "high_urgency_review_count"],
    ascending=[True, False, False],
)

print("최근성 구간별 이슈 요약:", recency_issue_summary.shape)
display(recency_issue_summary.head(20))


최근성 구간별 이슈 요약: (58, 7)


,review_recency_group,llm_issue_category,issue_name_kor,affected_review_count,negative_mixed_review_count,high_urgency_review_count,steam_negative_review_count
46,last_30d,gameplay_loop,게임플레이 루프,112,81,30,56
39,last_30d,balance,밸런스,55,50,19,29
47,last_30d,graphics_audio,그래픽/사운드,51,38,11,18
40,last_30d,bug,버그,38,33,13,16
56,last_30d,story,스토리,54,31,7,23
41,last_30d,content_volume,콘텐츠 분량,36,24,6,15
50,last_30d,other,기타,34,23,3,22
51,last_30d,performance,성능,25,22,13,13
45,last_30d,difficulty,난이도,26,21,8,12
57,last_30d,ui_ux,UI/UX,22,19,5,11


# 8. 04-2 LLM 입력용 근거 데이터 생성

04-2에서는 이 데이터를 LLM에게 제공해 **패치·운영 전략 초안**을 만들 예정이다.

핵심 원칙은 다음과 같다.

- LLM에게 개별 리뷰만 주고 최종 우선순위를 판단하게 하지 않는다.
- 04-1에서 만든 반복 이슈 근거, 부정·혼합 분포, Steam 비추천 맥락, 최근성, 플레이타임 근거를 함께 제공한다.
- `High urgency`는 04번 LLM 리뷰 분류에서 나온 보조 참고 지표로만 전달한다.
- 04-2의 LLM은 `action_group_hint`와 `rule_priority_hint`를 새로 판단하지 않고, 개발자가 읽기 쉬운 문장으로 정리하는 역할만 한다.


In [13]:
# ============================================================
# 04-2 LLM 입력용 근거 문장 생성 함수
# ============================================================

def clean_evidence_text(x, max_len=180):
    text = normalize_text_value(x)
    text = re.sub(r"\s+", " ", text)
    if len(text) > max_len:
        return text[:max_len].rstrip() + "..."
    return text


def collect_examples(issue_category, max_examples=5):
    """이슈별 대표 리뷰 요약/근거/개선 제안을 가져온다."""
    g = issue_review_base[issue_review_base["llm_issue_category"] == issue_category].copy()

    if len(g) == 0:
        return ""

    # 부정/혼합 + 최근 리뷰 + Steam 비추천 리뷰를 우선적으로 보여준다.
    # High urgency는 보조 참고 지표이므로 대표 리뷰 정렬에서 가장 앞 기준으로 쓰지 않는다.
    g["sort_negative"] = g["issue_negative_or_mixed_flag"].astype(int)
    g["sort_steam_negative"] = g["steam_negative_flag"].astype(int)
    g["sort_recent"] = g["recent_30d_flag"].astype(int)
    g["sort_high"] = g["high_urgency_flag"].astype(int)
    g["sort_votes"] = pd.to_numeric(g.get("votes_up", 0), errors="coerce").fillna(0)

    g = g.sort_values(
        ["sort_negative", "sort_steam_negative", "sort_recent", "sort_high", "sort_votes"],
        ascending=[False, False, False, False, False],
    )

    examples = []
    used = set()

    for _, row in g.iterrows():
        rec_id = row["recommendationid"]
        if rec_id in used:
            continue
        used.add(rec_id)

        evidence = clean_evidence_text(row.get("llm_issue_evidence", ""))
        summary = clean_evidence_text(row.get("llm_review_summary", ""))
        suggested = clean_evidence_text(row.get("llm_suggested_action", ""))

        example = (
            f"- steam_label={row.get('steam_label_text', '')}, "
            f"issue_sentiment={row.get('llm_issue_sentiment', '')}, "
            f"urgency_candidate={row.get('llm_urgency_candidate', '')}, "
            f"playtime={row.get('playtime_stage', '')}, "
            f"recency={row.get('review_recency_group', '')} | "
            f"근거: {evidence} | 요약: {summary} | LLM 개선 제안 후보: {suggested}"
        )
        examples.append(example)

        if len(examples) >= max_examples:
            break

    return "\n".join(examples)


def build_llm_evidence_text(row):
    """04-2 프롬프트에 바로 넣기 쉬운 이슈별 근거 블록을 만든다."""
    examples = collect_examples(row["llm_issue_category"], max_examples=5)

    return f"""
[ISSUE]
issue_category: {row['llm_issue_category']}
issue_name_kor: {row['issue_name_kor']}
action_group_hint: {row['action_group_hint']}
rule_priority_hint: {row['rule_priority_hint']}
priority_rule_detail: {row['priority_rule_detail']}
affected_review_count: {int(row['affected_review_count'])}
negative_mixed_review_count: {int(row['negative_mixed_review_count'])}
steam_negative_review_count: {int(row['steam_negative_review_count'])}
recent_30d_negative_mixed_review_count: {int(row['recent_30d_negative_mixed_review_count'])}
early_playtime_negative_mixed_review_count: {int(row['early_playtime_negative_mixed_review_count'])}
high_urgency_review_count: {int(row['high_urgency_review_count'])}  # 보조 참고 지표
high_urgency_rate: {row['high_urgency_rate']}  # 보조 참고 지표
priority_reason: {row['priority_reason']}
patch_ops_note: {row['patch_ops_note']}
대표 근거:
{examples}
[/ISSUE]
""".strip()


In [14]:
# ============================================================
# 04-2 LLM 입력용 근거 테이블 생성
# ============================================================

patch_ops_evidence_base = issue_summary.copy()
patch_ops_evidence_base["llm_evidence_text"] = patch_ops_evidence_base.apply(build_llm_evidence_text, axis=1)

# 04-2에서 너무 많은 이슈를 모두 넣지 않도록 기본 정렬 상태로 저장한다.
# 필요하면 04-2에서 상/중 우선순위만 필터링해서 사용할 수 있다.

print("패치·운영 전략 생성용 근거 테이블:", patch_ops_evidence_base.shape)
display(patch_ops_evidence_base[[
    "llm_issue_category", "issue_name_kor", "action_group_hint", "rule_priority_hint",
    "priority_rule_detail", "affected_review_count", "negative_mixed_review_count",
    "steam_negative_review_count", "high_urgency_review_count",
    "recent_30d_negative_mixed_review_count", "early_playtime_negative_mixed_review_count",
    "patch_ops_note",
]].head(20))


패치·운영 전략 생성용 근거 테이블: (21, 27)


,llm_issue_category,issue_name_kor,action_group_hint,rule_priority_hint,priority_rule_detail,affected_review_count,negative_mixed_review_count,steam_negative_review_count,high_urgency_review_count,recent_30d_negative_mixed_review_count,early_playtime_negative_mixed_review_count,patch_ops_note
1,bug,버그,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,95,85,41,38,33,11,"반복 언급된 버그를 재현 가능성 기준으로 분류하고, 플레이 방해 수준이 큰 항목부터 수정한다."
13,performance,성능,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,60,53,26,26,22,13,"프레임 저하, 로딩, 끊김 등 성능 문제를 환경별로 점검한다."
4,crash,크래시,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,30,29,25,26,11,7,"크래시 발생 조건과 로그를 우선 확인하고, 재현 가능한 오류부터 수정한다."
11,optimization,최적화,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,27,26,16,14,11,11,최적화 이슈가 특정 구간이나 사양에서 반복되는지 확인한다.
17,save_progression,저장/진행,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,27,24,23,23,8,3,"저장 손실, 진행 막힘, 퀘스트 진행 불가 여부를 우선 점검한다."
7,gameplay_loop,게임플레이 루프,단기 개선,상,부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복,284,208,150,73,81,41,"반복 피로, 목표 구조, 보상 흐름을 점검하고 플레이 루프의 지루함을 줄인다."
0,balance,밸런스,단기 개선,상,부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복,130,118,64,40,50,6,"전투, 성장, 보상, 적 난이도의 불균형 지점을 조정한다."
20,ui_ux,UI/UX,단기 개선,상,부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복,63,58,36,20,19,8,"메뉴, 인벤토리, 퀘스트 안내, 조작 안내 등 편의성 문제를 개선한다."
6,difficulty,난이도,단기 개선,상,부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복,68,48,30,19,21,7,초반 진입 장벽과 후반 난이도 피로를 구분해 난이도 옵션 또는 안내를 보강한다.
16,progression_grind,성장/반복 노가다,단기 개선,상,부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복,28,26,16,14,9,2,반복 성장과 노가다 피로를 줄일 수 있는 보상/성장 속도 조정을 검토한다.


# 9. Tableau/보고서용 원천 데이터 생성

이슈 태그 단위 원천 데이터에 이슈 요약값을 붙여, 대시보드에서 필터/집계를 하기 쉽게 만든다.


In [15]:
# ============================================================
# Tableau/보고서용 원천 데이터 생성
# ============================================================

summary_cols_for_merge = [
    "llm_issue_category", "action_group_hint", "rule_priority_hint",
    "affected_review_count", "negative_mixed_review_count", "high_urgency_review_count",
    "high_urgency_rate", "negative_mixed_rate", "patch_ops_note",
]

tableau_source = issue_review_base.merge(
    issue_summary[summary_cols_for_merge],
    on="llm_issue_category",
    how="left",
    suffixes=("", "_issue_summary"),
)

# Tableau에서 쓰기 좋은 컬럼만 선택
keep_cols = [
    "appid", "game_name", "recommendationid",
    "review_datetime", "release_date", "days_from_release", "release_period", "review_recency_group",
    "steam_label_text", "llm_sentiment", "steam_llm_sentiment_relation",
    "playtime_at_review_hours", "playtime_stage", "early_playtime_flag",
    "votes_up", "weighted_vote_score",
    "llm_urgency_candidate", "high_urgency_flag",
    "llm_issue_category", "issue_name_kor", "llm_issue_sentiment",
    "issue_positive_flag", "issue_negative_or_mixed_flag",
    "action_group_hint", "rule_priority_hint",
    "affected_review_count", "negative_mixed_review_count", "high_urgency_review_count",
    "high_urgency_rate", "negative_mixed_rate",
    "patch_ops_note", "llm_issue_evidence",
]
keep_cols = [col for col in keep_cols if col in tableau_source.columns]
tableau_source = tableau_source[keep_cols].copy()

print("Tableau/보고서 원천 데이터:", tableau_source.shape)
display(tableau_source.head())


Tableau/보고서 원천 데이터: (1967, 32)


,appid,game_name,recommendationid,review_datetime,release_date,days_from_release,release_period,review_recency_group,steam_label_text,llm_sentiment,steam_llm_sentiment_relation,playtime_at_review_hours,playtime_stage,early_playtime_flag,votes_up,weighted_vote_score,llm_urgency_candidate,high_urgency_flag,llm_issue_category,issue_name_kor,llm_issue_sentiment,issue_positive_flag,issue_negative_or_mixed_flag,action_group_hint,rule_priority_hint,affected_review_count,negative_mixed_review_count,high_urgency_review_count,high_urgency_rate,negative_mixed_rate,patch_ops_note,llm_issue_evidence
0,1466060,Tainted Grail: The Fall of Avalon,221779395,2026-03-26 21:36:35,2025-05-23,307.0,D181+,31-60d,positive,positive,exact_match,18.666667,5-20h,False,0,0.5,low,False,positive_praise,긍정 칭찬,positive,True,False,강점 유지,하,615,0,22,0.0358,0.0000,"긍정적으로 평가된 요소를 유지하고, 업데이트와 마케팅 메시지에서 강점으로 활용한다.",Absolute banger of a game. Scratches both the Elder Scrolls as well as the Gothic/PB itch perfectly.
1,1466060,Tainted Grail: The Fall of Avalon,222596326,2026-04-05 16:40:12,2025-05-23,317.0,D181+,last_30d,positive,positive,exact_match,27.050000,20-50h,False,0,0.5,low,False,gameplay_loop,게임플레이 루프,positive,True,False,단기 개선,상,284,208,73,0.2570,0.7324,"반복 피로, 목표 구조, 보상 흐름을 점검하고 플레이 루프의 지루함을 줄인다.","You can turn your hard-working summons... Into cheese... And then you can eat them, or sell them..."
2,1466060,Tainted Grail: The Fall of Avalon,219954559,2026-03-06 06:19:32,2025-05-23,287.0,D181+,31-60d,positive,positive,exact_match,20.383333,20-50h,False,1,0.5,low,False,positive_praise,긍정 칭찬,positive,True,False,강점 유지,하,615,0,22,0.0358,0.0000,"긍정적으로 평가된 요소를 유지하고, 업데이트와 마케팅 메시지에서 강점으로 활용한다.","Actually the best game ive played in years, love the art, love the gameplay"
3,1466060,Tainted Grail: The Fall of Avalon,217220762,2026-01-31 03:40:11,2025-05-23,253.0,D181+,61-90d,positive,positive,exact_match,52.250000,50h+,False,0,0.5,low,False,positive_praise,긍정 칭찬,positive,True,False,강점 유지,하,615,0,22,0.0358,0.0000,"긍정적으로 평가된 요소를 유지하고, 업데이트와 마케팅 메시지에서 강점으로 활용한다.","great game, great first game of this type for these devs"
4,1466060,Tainted Grail: The Fall of Avalon,224253174,2026-04-28 01:50:32,2025-05-23,340.0,D181+,last_30d,positive,positive,exact_match,9.533333,5-20h,False,1,0.5,low,False,positive_praise,긍정 칭찬,positive,True,False,강점 유지,하,615,0,22,0.0358,0.0000,"긍정적으로 평가된 요소를 유지하고, 업데이트와 마케팅 메시지에서 강점으로 활용한다.","love-child of Skyrim, Souls, and the grim-dark aesthetic"


# 10. CSV 저장

In [16]:
# ============================================================
# CSV 저장
# ============================================================
# 04-1 산출물은 04-2에서 실제로 사용할 최소 파일만 저장한다.

review_base.to_csv(POSTLAUNCH_REVIEW_BASE_PATH, index=False, encoding="utf-8-sig")
issue_summary.to_csv(POSTLAUNCH_ISSUE_SUMMARY_PATH, index=False, encoding="utf-8-sig")
patch_ops_evidence_base.to_csv(POSTLAUNCH_PATCH_OPS_EVIDENCE_BASE_PATH, index=False, encoding="utf-8-sig")
tableau_source.to_csv(TABLEAU_POSTLAUNCH_SOURCE_PATH, index=False, encoding="utf-8-sig")

saved_files = pd.DataFrame([
    {"파일명": POSTLAUNCH_REVIEW_BASE_PATH.name, "행 수": len(review_base), "역할": "리뷰 1개 단위 전처리 결과"},
    {"파일명": POSTLAUNCH_ISSUE_SUMMARY_PATH.name, "행 수": len(issue_summary), "역할": "이슈별 반복성/부정·혼합/High urgency 요약"},
    {"파일명": POSTLAUNCH_PATCH_OPS_EVIDENCE_BASE_PATH.name, "행 수": len(patch_ops_evidence_base), "역할": "04-2 LLM 패치·운영 전략 생성용 근거 데이터"},
    {"파일명": TABLEAU_POSTLAUNCH_SOURCE_PATH.name, "행 수": len(tableau_source), "역할": "Tableau/보고서용 원천 데이터"},
])

print("저장 완료")
print("저장 폴더:", POSTLAUNCH_PREPROCESS_DIR)
display(saved_files)

저장 완료
저장 폴더: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_tainted-grail\postlaunch_preprocess_data


,파일명,행 수,역할
0,postlaunch_review_base.csv,1000,리뷰 1개 단위 전처리 결과
1,postlaunch_issue_summary.csv,21,이슈별 반복성/부정·혼합/High urgency 요약
2,postlaunch_patch_ops_evidence_base.csv,21,04-2 LLM 패치·운영 전략 생성용 근거 데이터
3,tableau_postlaunch_patch_ops_source.csv,1967,Tableau/보고서용 원천 데이터


# 11. 우선순위 산정 로직 검증

In [ ]:
# ============================================================
# 우선순위 산정 로직 검증 (GPT)
# ============================================================
# 목적:
# 튜터님 피드백이 코드에 실제로 반영되었는지 확인한다.
# 특히 High urgency가 rule_priority_hint 계산에 직접 들어가지 않았는지,
# Steam 비추천/부정·혼합 근거 없이 '상' 우선순위가 부여되지 않았는지 점검한다.

import dis

# 1. 리뷰 단위 결과는 recommendationid가 중복되면 안 된다.
assert review_base["recommendationid"].is_unique, "review_base에 recommendationid 중복이 있습니다."

# 2. 이슈 요약의 affected_review_count는 전체 분석 리뷰 수보다 클 수 없다.
assert (issue_summary["affected_review_count"] <= review_base["recommendationid"].nunique()).all()

# 3. 부정·혼합 리뷰 수는 영향 리뷰 수보다 클 수 없다.
assert (issue_summary["negative_mixed_review_count"] <= issue_summary["affected_review_count"]).all()

# 4. High urgency 리뷰 수는 영향 리뷰 수보다 클 수 없다.
assert (issue_summary["high_urgency_review_count"] <= issue_summary["affected_review_count"]).all()

# 5. get_rule_priority_hint 함수 내부에서 high_urgency 계열 컬럼을 직접 참조하지 않는지 확인한다.
priority_func_names = set(get_rule_priority_hint.__code__.co_names)
priority_func_consts = {str(x) for x in get_rule_priority_hint.__code__.co_consts if isinstance(x, str)}
priority_func_refs = priority_func_names | priority_func_consts
high_refs = [x for x in priority_func_refs if "high_urgency" in x or x == "high"]
assert len(high_refs) == 0, f"rule_priority_hint 계산 함수에서 High urgency 참조가 발견되었습니다: {high_refs}"

# 6. '상' 우선순위는 부정·혼합 + Steam 비추천 근거가 함께 있어야 한다.
high_without_negative_context = issue_summary[
    (issue_summary["rule_priority_hint"] == "상")
    & (
        (issue_summary["negative_mixed_review_count"] < HIGH_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS)
        | (issue_summary["steam_negative_review_count"] < HIGH_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS)
    )
]
assert len(high_without_negative_context) == 0, high_without_negative_context[[
    "llm_issue_category", "issue_name_kor", "negative_mixed_review_count", "steam_negative_review_count", "rule_priority_hint"
]]

# 7. High urgency는 많지만 핵심 부정 근거가 약한데 '상'으로 올라간 이슈가 없어야 한다.
high_urgency_only_high = issue_summary[
    (issue_summary["rule_priority_hint"] == "상")
    & (issue_summary["high_urgency_review_count"] >= 10)
    & (issue_summary["negative_mixed_review_count"] < HIGH_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS)
]
assert len(high_urgency_only_high) == 0, high_urgency_only_high[[
    "llm_issue_category", "issue_name_kor", "high_urgency_review_count", "negative_mixed_review_count", "rule_priority_hint"
]]

# 8. 즉시 확인은 기술/진행 차단 가능성이 있는 이슈 그룹으로만 제한한다.
immediate_invalid = issue_summary[
    (issue_summary["action_group_hint"] == "즉시 확인")
    & (~issue_summary["llm_issue_category"].isin(IMMEDIATE_ISSUES))
]
assert len(immediate_invalid) == 0, immediate_invalid[[
    "llm_issue_category", "issue_name_kor", "action_group_hint"
]]

# 9. 강점 유지 항목은 개선 우선순위 '상'으로 들어가면 안 된다.
strength_high = issue_summary[
    (issue_summary["action_group_hint"] == "강점 유지")
    & (issue_summary["rule_priority_hint"] == "상")
]
assert len(strength_high) == 0, strength_high[[
    "llm_issue_category", "issue_name_kor", "action_group_hint", "rule_priority_hint"
]]

# 10. 04-2 입력용 근거 텍스트가 비어 있지 않은지 확인한다.
assert patch_ops_evidence_base["llm_evidence_text"].notna().all()

print("검증 통과")
print("리뷰 수:", review_base["recommendationid"].nunique())
print("리뷰-이슈 수:", len(issue_review_base))
print("이슈 종류 수:", issue_summary["llm_issue_category"].nunique())
print("High urgency는 rule_priority_hint 계산에는 직접 사용하지 않고, priority_reason/llm_evidence_text에서 보조 지표로만 유지한다.")


검증 통과
리뷰 수: 1000
리뷰-이슈 수: 1967
이슈 종류 수: 21
High urgency는 rule_priority_hint 계산에는 직접 사용하지 않고, priority_reason/llm_evidence_text에서 보조 지표로만 유지한다.


# 12. 산출 CSV 테이블 명세서

04-1에서는 04-2에서 실제로 사용할 산출물만 저장한다.

## 산출 파일 목록

| 파일명 | 역할 |
|---|---|
| `postlaunch_review_base.csv` | 리뷰 1개 단위 전처리 결과 |
| `postlaunch_issue_summary.csv` | 이슈별 반복성, 부정·혼합, Steam 비추천, 최근성, 플레이타임 근거 요약 |
| `postlaunch_patch_ops_evidence_base.csv` | 04-2 LLM 패치·운영 전략 생성용 근거 데이터 |
| `tableau_postlaunch_patch_ops_source.csv` | Tableau/보고서용 원천 데이터 |

---

## postlaunch_review_base.csv

| 컬럼명 | 설명 | 예시 값 |
|---|---|---|
| `recommendationid` | Steam 리뷰 고유 ID | `221779395` |
| `appid` | Steam 게임 고유 ID | `1466060` |
| `game_name` | 게임명 | `Tainted Grail: The Fall of Avalon` |
| `review_datetime` | 리뷰 작성 일시 | `2026-04-05 16:40:12` |
| `review_recency_group` | 분석 데이터 내 최신 리뷰일 기준 최근성 구간 | `last_30d` |
| `steam_label_text` | Steam 추천/비추천 라벨 | `positive` |
| `playtime_at_review_hours` | 리뷰 작성 시점 플레이타임 | `12.5` |
| `playtime_stage` | 리뷰 작성 시점 플레이타임 구간 | `5-20h` |
| `llm_sentiment` | LLM이 리뷰 내용을 보고 분류한 감정 | `mixed` |
| `llm_urgency_candidate` | LLM이 리뷰 내용을 보고 분류한 시급도 후보 | `high` |
| `high_urgency_flag` | High urgency 여부. 우선순위 계산 기준이 아니라 보조 참고 지표 | `True` |
| `llm_review_summary` | LLM이 작성한 리뷰 요약 | `전투와 탐험은 좋지만 버그를 지적함` |
| `llm_suggested_action` | LLM이 제안한 리뷰 단위 개선 방향 후보 | `진행 방해 버그를 우선 확인` |

---

## postlaunch_issue_summary.csv

| 컬럼명 | 설명 | 예시 값 |
|---|---|---|
| `llm_issue_category` | LLM이 추출한 이슈 카테고리 | `gameplay_loop` |
| `issue_name_kor` | 이슈 한글명 | `게임플레이 루프` |
| `affected_review_count` | 해당 이슈가 언급된 리뷰 수 | `239` |
| `negative_mixed_review_count` |  LLM이 이슈 태그 단위에서 부정 또는 혼합 맥락으로 분류한 리뷰 수. Steam 비추천 라벨과는 별개의 LLM 분류값 | `130` |
| `steam_negative_review_count` | Steam 비추천 리뷰 중 해당 이슈가 언급된 리뷰 수 | `64` |
| `high_urgency_review_count` | LLM이 High urgency 후보로 분류한 리뷰 수. 우선순위 계산에는 직접 사용하지 않음 | `42` |
| `high_urgency_rate` | 영향 리뷰 중 High urgency 후보 비율. 보조 참고 지표 | `0.176` |
| `negative_mixed_rate` | 영향 리뷰 중 부정/혼합 비율 | `0.544` |
| `steam_negative_rate` | 영향 리뷰 중 Steam 비추천 비율 | `0.269` |
| `recent_30d_negative_mixed_review_count` | 최근 30일 부정/혼합 리뷰 수 | `12` |
| `early_playtime_negative_mixed_review_count` | 0~5시간 구간 부정/혼합 리뷰 수 | `8` |
| `action_group_hint` | 이슈 성격과 집계값 기반 대응 구분 | `단기 개선` |
| `rule_priority_hint` | 규칙 기반 우선 검토 수준 | `상` |
| `priority_rule_detail` | 우선 검토 수준이 부여된 규칙 설명 | `부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복` |
| `priority_reason` | 우선 검토 힌트의 근거 요약 | `영향 리뷰 239개, 부정·혼합 130개...` |
| `patch_ops_note` | 패치·운영 해석 메모 | `반복 피로, 목표 구조, 보상 흐름을 점검...` |

---

## postlaunch_patch_ops_evidence_base.csv

| 컬럼명 | 설명 | 예시 값 |
|---|---|---|
| `llm_issue_category` | LLM이 추출한 이슈 카테고리 | `save_progression` |
| `issue_name_kor` | 이슈 한글명 | `저장/진행` |
| `action_group_hint` | 04-1에서 계산한 대응 구분 | `즉시 확인` |
| `rule_priority_hint` | 04-1에서 계산한 고정 우선 검토 수준 | `상` |
| `priority_rule_detail` | 우선 검토 수준이 부여된 규칙 설명 | `플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락` |
| `priority_reason` | 집계 기반 근거 문장 | `영향 리뷰 26개, 부정·혼합 23개...` |
| `patch_ops_note` | 이슈별 기본 대응 메모 | `저장 손실, 진행 막힘...` |
| `llm_evidence_text` | 04-2 LLM 프롬프트에 넣을 근거 블록 | `[ISSUE] ... [/ISSUE]` |

---

## tableau_postlaunch_patch_ops_source.csv

| 컬럼명 | 설명 | 예시 값 |
|---|---|---|
| `review_datetime` | 리뷰 작성 일시 | `2026-04-05 16:40:12` |
| `review_recency_group` | 최근성 구간 | `last_30d` |
| `playtime_stage` | 플레이타임 구간 | `20-50h` |
| `llm_issue_category` | 이슈 카테고리 | `performance` |
| `issue_name_kor` | 이슈 한글명 | `성능` |
| `llm_issue_sentiment` | 해당 이슈가 리뷰에서 나타난 감정 | `negative` |
| `issue_negative_or_mixed_flag` | 부정/혼합 맥락 여부 | `True` |
| `action_group_hint` | 대응 구분 힌트 | `즉시 확인` |
| `rule_priority_hint` | 우선 검토 힌트 | `상` |
| `priority_rule_detail` | 우선 검토 힌트 부여 규칙 | `부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복` |
| `high_urgency_rate` | 이슈별 High urgency 후보 비율. 보조 참고 지표 | `0.42` |
| `patch_ops_note` | 패치·운영 해석 메모 | `성능 저하와 프레임 드랍을 우선 확인...` |
